# Create and run a local RAG pipeline from scratch

## What is RAG?

**RAG** stands for Retrieval-Augmented Generation.

The goal of RAG is to take information and pass it to an LLM so it can generate outputs based on that info.

* **Retrieval** - Find relevant information given a query, e.g. "what are macronutrients?" -> retrieve passages of text related to macronutrients.
* **Augmented** - We want to take the relevant info from retrieval and augment our input (prompt) to an LLM with that information.
* **Generation** -  Take the first two steps and pass them to an LLM for generative outputs.

## Why RAG?

The main goal of RAG is to improve the generation outputs of LLMs.

1. Prevent hallucinations - LLMs are good at generation good looking text, but that does not mean that it is factually correct.
    * RAG can help LLMs generate information based on relevant passages that are factual
2. Work with custom data - Many base LLMs are trained with internet-scale data (means good understanding with language in general), but means the responses can be generic in nature.
    * RAG helps create specific responses based on specific documents

## Overview of the RAG pipeline

1. Open a PDF document.
2. Format the text of the PDF textbook ready for an embedding model.
3. Embed all the chunks of text in the textbook and turn them into numerical representations which can be stored for use later.
4. Build a retrieval that uses vector search to find relevant chunks of text given a query.
5. Create a prompt that incorporates the retrieved pieces of text.
6. Generate an answer to the query based on the passages of the textbook that were retrieved with an LLM.

**Steps 1-3:** Document preprocessing and embedding creation

**Steps 4-6:** Retrieval and generation (search and answer)

### 1. Document/text processing and embedding creation

Items needed:
* PDF document of choice
* Embedding moodel of choice

Steps:
1. Import the PDF document
2. Process text for embedding (e.g. split into chunks of sentences)
3. Embed text chunks with embedding model
4. Save embeddings to a file for later retrieval

### Import PDF document

In [2]:
import os # file path handling
import requests # download things off the internet

# Get PDF document path
pdf_path = "human-nutrition-text.pdf"

# Download pdf
if not os.path.exists(pdf_path):
    print(f"[INFO] File doesn't exist, downloading...")

    # URL of pdf
    url = "https://pressbooks.oer.hawaii.edu/humannutrition2/open/download?type=pdf"

    # local filename
    filename = pdf_path

    # Send a GET request to the URL
    response = requests.get(url)

    # Check if the request was successful
    if response.status_code == 200:
        # Open the file and save it
        with open(filename, "wb") as file:
            file.write(response.content)
        
        print(f"[INFO] The file has been downloaded and saved as {filename}")
    else:
        print(f"[ERR] Failed to download file. Status code: {response.status_code}")
else:
    print(f"[INFO] File {pdf_path} already exists.")

[INFO] File human-nutrition-text.pdf already exists.


### Open PDF

In [16]:
import fitz # PyMuPDF -> used to read pdf files
from tqdm.auto import tqdm # progress bars

def text_formatter(text: str) -> str:
    """Performs minor formatting on text"""
    
    # remove newlines and extra whitespace chars
    cleaned_text = text.replace("\n", " ").strip()

    return cleaned_text

def read_pdf(path: str) -> list[dict]:
    doc = fitz.open(path)

    pages = []

    # Loop through all the pages in the pdf
    for page_num, page in tqdm(enumerate(doc)):
        text = page.get_text() # get text of current page
        text = text_formatter(text) # format text

        # append all important info about the current page
        pages.append({
            "page_number" : page_num - 41,
            "char_count": len(text),
            "word_count": len(text.split(" ")),
            "sentence_count": len(text.split(". ")),
            "token_count": len(text) / 4, # 1 token ~= 4 chars
            "text": text
        })

    return pages

#### Key term

| Term | Description |
| ----- | ----- | 
| **Token** | A sub-word piece of text. For example, "hello, world!" could be split into ["hello", ",", "world", "!"]. A token can be a whole word,<br> part of a word or group of punctuation characters. 1 token ~= 4 characters in English, 100 tokens ~= 75 words.<br> Text gets broken into tokens before being passed to an LLM. |

In [19]:
pages = read_pdf(path=pdf_path)

import random

random.sample(pages, k=3)

0it [00:00, ?it/s]

[{'page_number': 543,
  'char_count': 795,
  'word_count': 154,
  'sentence_count': 5,
  'token_count': 198.75,
  'text': 'Image by  rawpixel.com  on  unsplash.co m / CC0  Vitamin E is found in many foods, especially those higher in fat,  such as nuts and oils. Some spices, such as paprika and red chili  pepper, and herbs, such as oregano, basil, cumin, and thyme, also  contain vitamin E. (Keep in mind spices and herbs are commonly  used in small amounts in cooking and therefore are a lesser source  of dietary vitamin E.) See Table 10.7 “Vitamin E Content of Various  Foods” for a list of foods and their vitamin E contents.  Everyday Connection  To increase your dietary intake of vitamin E from plant- based foods try a spinach salad with tomatoes and  sunflower seeds, and add a dressing made with sunflower  oil, oregano, and basil.  Table 9.7 Vitamin E Content of Various Foods  Fat-Soluble Vitamins  |  543'},
 {'page_number': 733,
  'char_count': 1581,
  'word_count': 255,
  'sentence_c